# Этап 2. Аудит данных Titanic

**Датасет:** `train.csv` из соревнования Titanic — Machine Learning from Disaster.  
**Задача:** бинарная классификация — предсказать, выжил пассажир или нет.  
**Целевая переменная:** `Survived`, где `1` — выжил, `0` — не выжил.

Цель этого ноутбука — проверить структуру, типы, пропуски, дубликаты, диапазоны и баланс классов до обучения модели.

## 1. Загрузка данных и библиотек

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

DATA_PATH = Path('data/titanic_train.csv')

# В репозитории файл уже лежит в папке data.
# Если ноутбук открыт отдельно в Google Colab, можно загрузить CSV вручную.
if not DATA_PATH.exists():
    DATA_PATH = Path('titanic_train.csv')

if not DATA_PATH.exists():
    from google.colab import files
    print('Загрузите файл titanic_train.csv')
    uploaded = files.upload()
    DATA_PATH = Path(next(iter(uploaded)))

df = pd.read_csv(DATA_PATH)
df.head()

## 2. Общая информация о датасете

In [ ]:
print(f'Строк: {df.shape[0]}')
print(f'Столбцов: {df.shape[1]}')
print(f'Одна строка: один пассажир')
print(f'Целевая переменная: Survived')

display(pd.DataFrame({
    'Столбец': df.columns,
    'Тип данных': df.dtypes.astype(str).values,
    'Непустых значений': df.notna().sum().values
}))

### Ожидаемые правила

- `PassengerId` — идентификатор пассажира, должен быть уникальным;
- `Survived` — только 0 или 1;
- `Pclass` — только 1, 2 или 3;
- `Age` — возраст, допустим положительный и не более 100 лет;
- `Fare` — стоимость билета, не может быть отрицательной.

## 3. Пропущенные значения

In [ ]:
missing = pd.DataFrame({
    'Пропусков': df.isna().sum(),
    'Доля, %': (df.isna().mean() * 100).round(2)
})
missing = missing[missing['Пропусков'] > 0].sort_values('Пропусков', ascending=False)
display(missing)

**Интерпретация:**

- `Cabin`: пропусков очень много, поэтому номер каюты лучше не заполнять «наугад». Для модели можно создать признак `HasCabin` — есть ли информация о каюте.
- `Age`: пропуски можно заполнить медианой, например в группах по полу и классу билета, и добавить отдельный признак `AgeMissing`.
- `Embarked`: пропусков мало; их можно заполнить наиболее частым портом посадки.

## 4. Дубликаты и ключи

In [ ]:
full_duplicates = int(df.duplicated().sum())
duplicate_passenger_id = int(df['PassengerId'].duplicated().sum())

duplicate_report = pd.DataFrame({
    'Проверка': ['Полные дубликаты строк', 'Повторы PassengerId'],
    'Нарушений': [full_duplicates, duplicate_passenger_id],
    'База': [len(df), len(df)],
    'Доля, %': [round(full_duplicates / len(df) * 100, 2), round(duplicate_passenger_id / len(df) * 100, 2)]
})
display(duplicate_report)

## 5. Проверка диапазонов и допустимых значений

In [ ]:
age_numeric = df['Age'].dropna()

checks = pd.DataFrame({
    'Проверка': [
        'PassengerId не уникален',
        'Survived не равен 0 или 1',
        'Pclass не входит в 1–3',
        'Возраст не в диапазоне 0–100',
        'Стоимость билета отрицательная'
    ],
    'Нарушений': [
        int(df['PassengerId'].duplicated().sum()),
        int((~df['Survived'].isin([0, 1])).sum()),
        int((~df['Pclass'].isin([1, 2, 3])).sum()),
        int(((age_numeric < 0) | (age_numeric > 100)).sum()),
        int((df['Fare'] < 0).sum())
    ],
    'База': [len(df), len(df), len(df), len(age_numeric), len(df)]
})
checks['Доля, %'] = (checks['Нарушений'] / checks['База'] * 100).round(2)
display(checks)

## 6. Уникальные и константные признаки

In [ ]:
unique_counts = pd.DataFrame({
    'Уникальных непустых значений': df.nunique(dropna=True)
}).sort_values('Уникальных непустых значений')
display(unique_counts)

constant_columns = unique_counts[unique_counts['Уникальных непустых значений'] <= 1]
print('Константные признаки:' if not constant_columns.empty else 'Константных признаков нет.')
display(constant_columns)

## 7. Баланс классов

In [ ]:
class_counts = df['Survived'].value_counts(dropna=False).sort_index()
class_profile = pd.DataFrame({
    'Класс': class_counts.index.map({0: 'Не выжил', 1: 'Выжил'}),
    'Количество': class_counts.values,
    'Доля, %': (class_counts.values / len(df) * 100).round(2)
})
display(class_profile)

ax = class_profile.plot.bar(x='Класс', y='Количество', legend=False, color=['#8c8c8c', '#4f81bd'], figsize=(6, 4))
ax.set_title('Распределение классов Survived')
ax.set_xlabel('Класс')
ax.set_ylabel('Количество пассажиров')
plt.xticks(rotation=0)
plt.show()

**Вывод по балансу классов:** 61,62% пассажиров не выжили и 38,38% выжили. Дисбаланс умеренный, поэтому при разбиении данных нужно использовать стратификацию, а качество модели оценивать не только по Accuracy, но и по Precision, Recall и F1-score.

## 8. Примеры связей с целевой переменной

In [ ]:
survival_by_sex = (df.groupby('Sex')['Survived']
                   .agg(Пассажиров='count', Выжили='sum', Доля_выживших='mean')
                   .assign(Доля_выживших=lambda x: (x['Доля_выживших'] * 100).round(2)))
display(survival_by_sex)

survival_by_class = (df.groupby('Pclass')['Survived']
                     .agg(Пассажиров='count', Выжили='sum', Доля_выживших='mean')
                     .assign(Доля_выживших=lambda x: (x['Доля_выживших'] * 100).round(2)))
display(survival_by_class)

## 9. Профиль качества данных

In [ ]:
profile = pd.DataFrame({
    'Проверка': [
        'Пропуски возраста',
        'Пропуски каюты',
        'Пропуски порта посадки',
        'Полные дубликаты сверх первой строки',
        'Повторы PassengerId',
        'Возраст вне 0–100',
        'Класс билета вне 1–3',
        'Пропуски целевой метки Survived'
    ],
    'Число / база': [
        f"{df['Age'].isna().sum()} / {len(df)}",
        f"{df['Cabin'].isna().sum()} / {len(df)}",
        f"{df['Embarked'].isna().sum()} / {len(df)}",
        f"{df.duplicated().sum()} / {len(df)}",
        f"{df['PassengerId'].duplicated().sum()} / {len(df)}",
        f"{((age_numeric < 0) | (age_numeric > 100)).sum()} / {len(age_numeric)}",
        f"{(~df['Pclass'].isin([1, 2, 3])).sum()} / {len(df)}",
        f"{df['Survived'].isna().sum()} / {len(df)}"
    ],
    'Доля, %': [
        round(df['Age'].isna().mean() * 100, 2),
        round(df['Cabin'].isna().mean() * 100, 2),
        round(df['Embarked'].isna().mean() * 100, 2),
        round(df.duplicated().mean() * 100, 2),
        round(df['PassengerId'].duplicated().mean() * 100, 2),
        round((((age_numeric < 0) | (age_numeric > 100)).mean()) * 100, 2),
        round((~df['Pclass'].isin([1, 2, 3])).mean() * 100, 2),
        round(df['Survived'].isna().mean() * 100, 2)
    ]
})
display(profile)

## 10. Два приоритета обработки

1. **Каюта (`Cabin`) — 77,10% пропусков.** Не заполнять выдуманными номерами. Создать признак наличия каюты или исключить исходный столбец из модели.
2. **Возраст (`Age`) — 19,87% пропусков.** Заполнить медианой внутри групп по полу и классу билета; сохранить индикатор, что возраст отсутствовал.

Дополнительно при обучении модели использовать стратифицированное разбиение по `Survived` и оценивать результат по Accuracy, Precision, Recall и F1-score.

## 11. Итог аудита

Датасет пригоден для учебной задачи классификации. Полных дубликатов и нарушений основных диапазонов не выявлено, а целевая метка заполнена для всех строк. Главные ограничения — большое число пропусков в `Cabin` и заметная доля пропусков в `Age`; их нужно обработать до обучения модели.